### Convert data production coffee to CSV

In [5]:
import pandas as pd
import os
import glob

def extract_year_data(filepath):
    year = int(os.path.basename(filepath).replace('.xlsx', ''))
    df = pd.read_excel(filepath, header=None)
    
    # Safely filter rows where column 4 matches 'Kopi Arabika'
    mask = df[4].astype(str).str.strip() == 'Kopi Arabika'
    data = df[mask].copy()
    
    # Clean village names
    data['village'] = data[3].astype(str).str.strip()
    
    # Handle Indonesian thousand separators (commas) and coerce to numeric
    data['tm_ha'] = pd.to_numeric(data[6].astype(str).str.replace(',', ''), errors='coerce')
    data['produksi_kg'] = pd.to_numeric(data[10].astype(str).str.replace(',', ''), errors='coerce')
    
    result = pd.DataFrame({
        'year': year,
        'village': data['village'],
        'tm_ha': data['tm_ha'],
        'produksi_kg': data['produksi_kg']
    })
    return result

# Process all xlsx files
files = sorted(glob.glob('../data/*.xlsx'))
print(f"Found files: {files}")

all_data = pd.concat([extract_year_data(f) for f in files], ignore_index=True)

# Sort deterministically within each year to ensure consistent numbering
all_data = all_data.sort_values(['year', 'village']).reset_index(drop=True)

# Count occurrences of each village per year
dup_counts = all_data.groupby(['year', 'village'])['village'].transform('count')

# Generate sequential counter (1, 2, 3...) for each village within its year
seq_numbers = all_data.groupby(['year', 'village']).cumcount() + 1

# Append number ONLY if the village appears more than once in that specific year
is_duplicate = dup_counts > 1
all_data.loc[is_duplicate, 'village'] = (
    all_data.loc[is_duplicate, 'village'] + ' ' + seq_numbers[is_duplicate].astype(str)
)

# Optional: Create a stable numeric ID for the newly formatted village names
all_data['village_id'] = pd.Categorical(all_data['village']).codes + 1

# Final column order
all_data = all_data[['village_id', 'year', 'village', 'tm_ha', 'produksi_kg']]

all_data.to_csv('../data/coffeeProduction_benerMeriah.csv', index=False)

print(f"Total records : {len(all_data)}")
print(f"Years         : {sorted(all_data['year'].unique())}")
print(f"Villages      : {all_data['village'].nunique()} unique")
print(all_data.head(10))

Found files: ['../data/2021.xlsx', '../data/2022.xlsx', '../data/2023.xlsx', '../data/2024.xlsx', '../data/2025.xlsx']
Total records : 1100
Years         : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Villages      : 220 unique
   village_id  year         village   tm_ha  produksi_kg
0           1  2021       Alam Jaya  102.16     67668.31
1           2  2021     Alur Cincin   48.75     35453.23
2           3  2021     Alur Gading   27.63     18172.32
3           4  2021            Amor  421.39    341715.78
4           5  2021         Ayu Ara  178.86    127369.09
5           6  2021      Babussalam   11.07      9243.17
6           7  2021  Bahgie Bertona   11.78     11461.71
7           8  2021        Bale Atu  111.77     96083.04
8           9  2021     Bale Musara  208.00    153770.82
9          10  2021    Bale Purnama  272.31    207502.19


### Create manual calculation with excel
The formula have some issues due to difference in raw data and excel format

### For coffee_yield_pipeline.xlsx

In [2]:
# AUTO FETCH
import joblib
import pandas as pd

# Configuration
EXCEL_FILE = '../../code/coffee_yield_pipeline.xlsx'
SHEET_NAME='INFERENCE'
MODEL_PATH = '../result/best_model.pkl'
FEATURES_PATH = '../result/features.csv'

# Feature names in EXACT order (D44:D94)
FEATURE_NAMES = [
    'rainfall_mm','temperature_celsius','relative_humidity_percent',
    'soil_moisture_percent','wind_speed_10m','dtr_celsius','vpd_kpa',
    'net_solar_rad_kwh_m2','rainfall_mm_Q1','temperature_celsius_Q1',
    'relative_humidity_percent_Q1','soil_moisture_percent_Q1',
    'wind_speed_10m_Q1','dtr_celsius_Q1','vpd_kpa_Q1',
    'net_solar_rad_kwh_m2_Q1','rainfall_mm_Q2','temperature_celsius_Q2',
    'relative_humidity_percent_Q2','soil_moisture_percent_Q2',
    'wind_speed_10m_Q2','dtr_celsius_Q2','vpd_kpa_Q2',
    'net_solar_rad_kwh_m2_Q2','rainfall_mm_Q3','temperature_celsius_Q3',
    'relative_humidity_percent_Q3','soil_moisture_percent_Q3',
    'wind_speed_10m_Q3','dtr_celsius_Q3','vpd_kpa_Q3',
    'net_solar_rad_kwh_m2_Q3','rainfall_mm_Q4','temperature_celsius_Q4',
    'relative_humidity_percent_Q4','soil_moisture_percent_Q4',
    'wind_speed_10m_Q4','dtr_celsius_Q4','vpd_kpa_Q4',
    'net_solar_rad_kwh_m2_Q4','temperature_celsius_x_vpd_kpa',
    'temperature_celsius_x_soil_moisture_percent',
    'temperature_celsius_x_dtr_celsius',
    'temperature_celsius_x_rainfall_mm',
    'vpd_kpa_x_soil_moisture_percent','vpd_kpa_x_dtr_celsius',
    'vpd_kpa_x_rainfall_mm','soil_moisture_percent_x_dtr_celsius',
    'soil_moisture_percent_x_rainfall_mm',
    'dtr_celsius_x_rainfall_mm','prev_yield_kg_ha'
]

def main():
    model = joblib.load(MODEL_PATH)
    feats = pd.read_csv(FEATURES_PATH)['feature'].tolist()
    
    try:
        df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME, header=None)
    except ValueError as e:
        print(f"{e}")
        return
    
    # Extract values from column D (index 3), rows 44-94 (index 43-93)
    data = pd.DataFrame([df.iloc[43:94, 3].tolist()], columns=FEATURE_NAMES)
    
    # Check for missing/NaN values
    missing = data.columns[data.isna().any()].tolist()
    if missing:
        print(f"Warning!!! Fill the excel first!")
        print(f"{len(missing)} missing features: {missing}")
        return
    
    print(f"log_yield = {model.predict(data.reindex(columns=feats))[0]:.6f}")
    
if __name__ == "__main__":
    main()

log_yield = 6.498081
